In [1]:
!apt-get update -qq
!apt-get install -y --no-install-recommends \
    libgl1 \
    libglu1-mesa \
    libxrender1 \
    libxcursor1 \
    libxft2 \
    libxinerama1

!pip uninstall -y dolfinx gmsh >/dev/null 2>&1 || true
!pip install "numpy<2" scipy matplotlib numba pyyaml pillow gmsh mpi4py

!wget "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O "/tmp/fenicsx-install-release-real.sh"
!bash "/tmp/fenicsx-install-release-real.sh"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1 is already the newest version (1.7.0-1build1).
libglu1-mesa is already the newest version (9.0.2-1.1build1).
libxrender1 is already the newest version (1:0.9.10-1.1build1).
libxcursor1 is already the newest version (1:1.2.1-1build1).
libxft2 is already the newest version (2.3.6-1build1).
libxinerama1 is already the newest version (2:1.1.4-3build1).
0 upgraded, 0 newly installed, 0 to remove and 47 not upgraded.
  Using cached gmsh-4.15.2-py2.py3-none-manylinux_2_24_x86_64.whl.metadata (1.7 kB)
Using cached gmsh-4.15.2-py2.py3-none-manylinux_2_24_x86_64.whl (40.0 MB)
--2026-10-14 16:33:55--  https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh
Resolving fem-on-colab.github.io (fem

In [2]:
import sys
print(sys.version)
import gmsh
import dolfinx
print(dolfinx.__file__)
import gmshio
print("imports ok")

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
/usr/local/lib/python3.13/dist-packages/dolfinx/__init__.py


ModuleNotFoundError: No module named 'gmshio'

In [3]:
!pip install gmshio

In [ ]:
# %%

import gmsh
import numpy as np
from mpi4py import MPI
from dolfinx import fem, io, mesh
from dolfinx.io import gmshio
import ufl
from petsc4py import PETSc

# ==========================================
# 1. Mesh Generation in Gmsh
# ==========================================
gmsh.initialize()
gmsh.model.add("skyscraper_mfg")
occ = gmsh.model.occ

# Bounding Air Box & 4 Skyscrapers
air_box = occ.addBox(0, 0, 0, 100.0, 100.0, 50.0)
b1 = occ.addBox(15, 15, 0, 20, 20, 35)
b2 = occ.addBox(60, 15, 0, 25, 15, 40)
b3 = occ.addBox(15, 60, 0, 15, 25, 30)
b4 = occ.addBox(55, 55, 0, 25, 25, 45)

occ.cut([(3, air_box)], [(3, b1), (3, b2), (3, b3), (3, b4)])
occ.synchronize()

gmsh.option.setNumber("Mesh.MeshSizeMin", 3.0)
gmsh.option.setNumber("Mesh.MeshSizeMax", 10.0)

# Physical Groups
vol_entities = gmsh.model.getEntities(dim=3)
gmsh.model.addPhysicalGroup(3, [e[1] for e in vol_entities], tag=1, name="air_domain")
surf_entities = gmsh.model.getEntities(dim=2)
gmsh.model.addPhysicalGroup(2, [s[1] for s in surf_entities], tag=2, name="boundaries")

gmsh.model.mesh.generate(3)

# Pipeline directly into FEniCSx
domain, cell_tags, facet_tags = gmshio.model_to_mesh(
    gmsh.model, comm=MPI.COMM_WORLD, rank=0, gdim=3
)
gmsh.finalize()

# ==========================================
# 2. Time & Problem Parameters
# ==========================================
T_final = 2.0         # Time horizon [0, T]
N_steps = 20          # Number of time steps
dt = T_final / N_steps
nu = 0.5              # Agent diffusion coefficient
congestion_coef = 0.5 # Coupling weight for density penalty
max_mfg_iter = 10     # Outer fixed-point loop iterations
alpha_relax = 0.3     # Picard relaxation parameter

V = fem.functionspace(domain, ("Lagrange", 1))

# Storage across time horizon
u_traj = [fem.Function(V) for _ in range(N_steps + 1)]
m_traj = [fem.Function(V) for _ in range(N_steps + 1)]

# Initial Swarm Density m(0, x): Gaussian distribution near (10, 10, 10)
x_coord = ufl.SpatialCoordinate(domain)
m0_expr = ufl.exp(-0.02 * ((x_coord[0]-10)**2 + (x_coord[1]-10)**2 + (x_coord[2]-10)**2))
m0_func = fem.Expression(m0_expr, V.element.interpolation_points())

for n in range(N_steps + 1):
    m_traj[n].interpolate(m0_func)

# Terminal Value u(T, x): Target destination near (85, 85, 25)
uT_expr = 0.01 * ((x_coord[0]-85)**2 + (x_coord[1]-85)**2 + (x_coord[2]-25)**2)
uT_func = fem.Expression(uT_expr, V.element.interpolation_points())


# ==========================================
# 3. Solvers Setup
# ==========================================

# --- HJB Backward Step (Semi-Implicit Euler with Linearized Hamilton) ---
def solve_hjb_step(u_next, m_curr, dt_val, nu_val):
    u_curr = fem.Function(V)
    u_curr.x.array[:] = u_next.x.array[:]  # Warm start
    
    u_trial = ufl.TrialFunction(V)
    v_test = ufl.TestFunction(V)
    
    # Running cost: f(x, m) = congestion_coef * m
    f_cost = congestion_coef * m_curr
    
    # Linearized HJB variational form:
    # (u_curr - u_next)/dt - nu*Δ(u_curr) + 0.5*|∇u_curr|^2 = f
    a = (1.0 / dt_val) * u_trial * v_test * ufl.dx + nu_val * ufl.inner(ufl.grad(u_trial), ufl.grad(v_test)) * ufl.dx
    L = (1.0 / dt_val) * u_next * v_test * ufl.dx + f_cost * v_test * ufl.dx
    
    # Subtract non-linear gradient kinetic energy term explicitly/semi-implicitly
    L -= 0.5 * ufl.inner(ufl.grad(u_next), ufl.grad(u_next)) * v_test * ufl.dx

    problem = fem.petsc.LinearProblem(a, L, petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    return problem.solve()


# --- Fokker-Planck Forward Step (Advection-Diffusion) ---
def solve_fp_step(m_prev, u_curr, dt_val, nu_val):
    m_next = fem.Function(V)
    
    m_trial = ufl.TrialFunction(V)
    v_test = ufl.TestFunction(V)
    
    # Drift vector v_drift = -∇u
    v_drift = -ufl.grad(u_curr)
    
    # FP Variational form:
    # (m_next - m_prev)/dt - nu*Δ(m_next) - ∇·(m_next * v_drift) = 0
    a = (1.0 / dt_val) * m_trial * v_test * ufl.dx \
        + nu_val * ufl.inner(ufl.grad(m_trial), ufl.grad(v_test)) * ufl.dx \
        - ufl.inner(m_trial * v_drift, ufl.grad(v_test)) * ufl.dx

    L = (1.0 / dt_val) * m_prev * v_test * ufl.dx

    problem = fem.petsc.LinearProblem(a, L, petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    return problem.solve()


# ==========================================
# 4. Outer MFG Coupling Loop
# ==========================================
print("Starting Mean Field Game Coupled Forward-Backward Iterations...")

for it in range(max_mfg_iter):
    # --- Step A: Backward HJB Pass (from t=T down to t=0) ---
    u_traj[N_steps].interpolate(uT_func)
    
    for n in range(N_steps - 1, -1, -1):
        u_traj[n] = solve_hjb_step(u_traj[n + 1], m_traj[n + 1], dt, nu)
        
    # --- Step B: Forward Fokker-Planck Pass (from t=0 up to t=T) ---
    m_new_traj = [fem.Function(V) for _ in range(N_steps + 1)]
    m_new_traj[0].interpolate(m0_func)
    
    for n in range(0, N_steps):
        m_new_traj[n + 1] = solve_fp_step(m_new_traj[n], u_traj[n + 1], dt, nu)
        
    # --- Step C: Convergence Check & Relaxation ---
    diff = np.linalg.norm(m_new_traj[N_steps].x.array - m_traj[N_steps].x.array)
    print(f"Iteration {it+1:02d}/{max_mfg_iter:02d} | Density Trajectory Change L2: {diff:.6e}")
    
    # Relax update to stabilize Picard iterations
    for n in range(1, N_steps + 1):
        m_traj[n].x.array[:] = (1.0 - alpha_relax) * m_traj[n].x.array[:] + alpha_relax * m_new_traj[n].x.array[:]
        
    if diff < 1e-4:
        print("MFG System Converged Successfully!")
        break

# ==========================================
# 5. Export Time-Series Results to VTX / ParaView
# ==========================================
with io.VTXWriter(domain.comm, "mfg_swarm_density.bp", m_traj, engine="BP4") as vtx_m:
    for n in range(N_steps + 1):
        vtx_m.write(n * dt)

with io.VTXWriter(domain.comm, "mfg_value_function.bp", u_traj, engine="BP4") as vtx_u:
    for n in range(N_steps + 1):
        vtx_u.write(n * dt)

print("Exported full temporal MFG trajectories (mfg_swarm_density.bp & mfg_value_function.bp) for ParaView visualization!")
# %%
